In [ ]:
import gymnasium as gym
import numpy as np
from gymnasium import spaces
import math


class SimplexProjectionEnv(gym.Env):

    def __init__(
            self,
            n_hours = 24,             # Number of time steps in an episode
            alpha = 0.5,              # Efficiency factor for P1
            beta = 0.5,               # Efficiency factor for P2
            lambda_unmet = 100,     # Penalty weight for unmet demand
            lambda_curt = 50,

            # Battery parameters
            max_soc = 50.00,           # Max battery State of Charge
            min_soc = 0.1,             # Minimum battery State of Charge
            max_charge = 30.00,        # Max charge at instant t
            max_discharge = 30.00     # Max discharge at instant t
            ):
        super().__init__()
        self.n_hours = int(n_hours)
        self.a = float(alpha)
        self.b = float(beta)
        self.lambda_unmet = float(lambda_unmet)
        self.lambda_curt = float(lambda_curt)
        self.max_soc = float(max_soc)
        self.min_soc = float(min_soc)
        self.max_soc = float(max_soc)
        self.max_charge = float(max_charge)
        self.max_discharge = float(max_discharge)

        # Spaces
        self.action_space = spaces.Box(
            low=np.zeros(8, dtype=np.float32),
            high=np.ones(8, dtype=np.float32),
            dtype=np.float32,
        ) # [E11, E12, E21, E22, B1, B2] (normalized)

        self.observation_space = spaces.Box(
            low=np.zeros(6, dtype=np.float32),
            high=np.array([1.0, 200.0, 200.0, 20.0, 20.0, 100.00], dtype=np.float32),
            dtype=np.float32,
        ) # [t_norm, solar, wind, demand_1, demand_2, battery's soc]


    

    # Reset generates a new random episode with stochastic profiles
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.t = 0

        rng = self.np_random

        T = range(self.n_hours)
            
        # Generate random demand scale and energy profiles
        demand_scale, solar, wind = self.generate_energy_profiles(rng, T)

        # Generate random demand profiles
        demand_1, demand_2 = self.generate_demands(rng, T, demand_scale)

        # Generate random intial state of charge for the battery
        initial_soc = rng.uniform(self.min_soc, self.max_soc)

        # Clip profiles to ensure non-negativity
        solar = np.clip(solar, 0.0, None)
        wind = np.clip(wind, 0.0, None)
        demand_1 = np.clip(demand_1, 0.0, None)
        demand_2 = np.clip(demand_2, 0.0, None)
        

        self.solar = solar
        self.wind = wind
        self.demand_1 = demand_1
        self.demand_2 = demand_2
        self.soc = initial_soc

        t_norm = 0.0
        obs = np.array([t_norm, self.solar[self.t], self.wind[self.t], self.demand_1[self.t], self.demand_2[self.t], self.soc], dtype=np.float32)
        return obs, {}

    
    def step(self, action):


        # Get current profiles
        available_solar = float(self.solar[self.t])
        available_wind = float(self.wind[self.t]) 
        demand_1 = float(self.demand_1[self.t])   
        demand_2 = float(self.demand_2[self.t])  
        soc = float(self.soc)


        # Parse action
        U11, U12, U21, U22, U1B, U2B, UB1, UB2 = [float(x) for x in np.clip(action, 0.0, 1.0)]

        # Calculate energies
        E11 = U11 * available_solar                     # solar -> Electrolyser
        E12 = U12 * (available_solar - E11)             # solar -> ASU  
        E1B = U1B * (available_solar - E11 - E12)       # Solar -> Battery
        E21 = U21 * available_wind                      # Wind  -> Electroliser
        E22 = U22 * (available_wind - E21)              # Wind  -> ASU
        E2B = U2B * (available_wind - E21 - E22)        # Wind  -> Battery
        B1  = UB1 * soc                                 # Battery -> Electroliser
        B2  = UB2 * (soc - B1)                          # Battery -> ASU


                

        # Calculate final production
        P1 = self.a * (E11 + E21 + B1)
        P2 = self.b * (E12 + E22 + B2)

        # Calculate curtailment (unused renewable energy)
        curt_1 = max(0.0, available_solar - (E11+E12+E1B))
        curt_2 = max(0.0, available_wind - (E21+E22+E2B))

        # Calculate  unmet demand
        unmet_1 = max(0.0, demand_1 - P1)
        unmet_2 = max(0.0, demand_2 - P2)

        # Reward: production minus penalties
        reward = P1 + P2 - (self.lambda_unmet * (unmet_1 + unmet_2)) - (self.lambda_curt * (curt_1 + curt_2))

        info = {
            "t": self.t,
            "solar": available_solar,
            "wind": available_wind,
            "demand_1": demand_1,
            "demand_2": demand_2,
            "E11": E11,
            "E12": E12,
            "E21": E21,
            "E22": E22,
            "E1B": E1B,
            "E2B": E2B,
            "B1" : B1,
            "B2" : B2,
            "P1" : P1,
            "P2" : P2,
        }

        # Advance time
        self.t += 1
        terminated = (self.t >= self.n_hours)
        truncated = False

        if terminated:
            obs = np.zeros(6, dtype=np.float32)
        else:
            t_norm = self.t / (self.n_hours - 1)
            soc = soc + (E1B + E2B) - (B1+B2)
            obs = np.array([t_norm, self.solar[self.t], self.wind[self.t], self.demand_1[self.t], self.demand_2[self.t], soc], dtype=np.float32)

        return obs, float(reward), terminated, truncated, info
    

    def generate_demands(self, rng, T, demand_scale):
        demand_1 = np.array([5.0 + 1.0 * math.sin(2 * math.pi * t / self.n_hours) for t in T], dtype=np.float32)
        demand_2 = np.array([4.0 + 1.0 * math.cos(2 * math.pi * t / self.n_hours) for t in T], dtype=np.float32)

        demand_1 *= demand_scale * (1.0 + rng.normal(0.0, 0.02, size=self.n_hours)).astype(np.float32)
        demand_2 *= demand_scale * (1.0 + rng.normal(0.0, 0.02, size=self.n_hours)).astype(np.float32)
        return demand_1,demand_2

    def generate_energy_profiles(self, rng, T):
        solar_peak = rng.uniform(40.0, 80.0)
        cloud_factor = rng.uniform(0.2, 1.0)
        wind_mean = rng.uniform(20.0, 50.0)
        wind_sigma = rng.uniform(2.0, 8.0)
        demand_scale = rng.uniform(0.85, 1.15)

            # Solar base shape (triangular with peak at midday)
        solar = np.zeros(self.n_hours, dtype=np.float32)
        for t in T:
            if 6 <= t < 18:
                solar[t] = solar_peak * (1.0 - abs(t - 12) / 6.0)
            # Apply cloudiness + noise
        solar *= cloud_factor
        solar *= (1.0 + rng.normal(0.0,0.05, size=self.n_hours)).astype(np.float32)
  
            # Wind: mean + noise
        wind = (wind_mean + rng.normal(0.0, wind_sigma, size=self.n_hours)).astype(np.float32)
        return demand_scale,solar,wind